<div style="background:linear-gradient(135deg,#0f0c29,#302b63);padding:20px 24px;border-radius:10px;border-left:5px solid #84cc16;font-family:Arial,sans-serif;">
  <h2 style="margin:0;color:#84cc16;">🥑&nbsp;Avocado Price Forecasting</h2>
  <p style="margin:8px 0 0;color:#bbb;font-size:14px;">EDA of US avocado prices 2015-2018 · trend & seasonality analysis · Prophet time-series forecast</p>
</div>

## Overview

Analysis and forecasting of US avocado prices using the [Hass Avocado Board dataset](https://www.kaggle.com/datasets/neuromusic/avocado-prices).

| Step | Tool |
|------|------|
| EDA & visualization | pandas, seaborn, plotly |
| Time-series decomposition | seasonal_decompose |
| Forecasting | Prophet (Meta) |

**Dataset:** `avocado.csv` — included in this branch.

```bash
pip install pandas matplotlib seaborn plotly prophet
```

## 1. Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("avocado.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Regions: {df['region'].nunique()} unique")
print(f"\nAveragePrice stats:")
print(df["AveragePrice"].describe().round(2))
df.head(3)

## 2. Price Distribution & Type Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Avocado Price Overview", fontsize=14, fontweight="bold")

# Price distribution
df["AveragePrice"].hist(bins=40, color="#84cc16", edgecolor="white", ax=axes[0])
axes[0].set_title("Price Distribution")
axes[0].set_xlabel("Average Price ($)")

# Conventional vs Organic
sns.boxplot(x="type", y="AveragePrice", data=df,
            palette={"conventional":"#86efac","organic":"#4ade80"}, ax=axes[1])
axes[1].set_title("Conventional vs Organic")

# Price by year
df_yr = df.groupby("year")["AveragePrice"].mean()
df_yr.plot(kind="bar", color="#84cc16", ax=axes[2], rot=0)
axes[2].set_title("Average Price by Year")
axes[2].set_xlabel("")

plt.tight_layout()
plt.show()

## 3. Top 10 Regions by Average Price

In [ ]:
top_regions = (
    df.groupby("region")["AveragePrice"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
top_regions.plot(kind="barh", color="#84cc16", ax=ax)
ax.set_title("Top 10 Most Expensive Avocado Markets", fontsize=13, fontweight="bold")
ax.set_xlabel("Average Price ($)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. National Price Trend (All Regions Combined)

In [ ]:
national = (
    df[df["region"] == "TotalUS"]
    .groupby("Date")["AveragePrice"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(national["Date"], national["AveragePrice"], color="#84cc16", linewidth=1.8)
ax.fill_between(national["Date"], national["AveragePrice"],
                national["AveragePrice"].min(), alpha=0.1, color="#84cc16")
ax.set_title("National Average Avocado Price Over Time", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Average Price ($)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Prophet Forecast — Next 52 Weeks

In [ ]:
from prophet import Prophet

# Prepare data: Prophet expects columns ds (date) and y (value)
prophet_df = national.rename(columns={"Date": "ds", "AveragePrice": "y"})

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.1,
)
model.fit(prophet_df)

future   = model.make_future_dataframe(periods=52, freq="W")
forecast = model.predict(future)

print(f"Forecast generated: {len(forecast)} rows")
forecast[["ds","yhat","yhat_lower","yhat_upper"]].tail(10)

## 6. Forecast Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full forecast
axes[0].plot(prophet_df["ds"], prophet_df["y"],
             color="#84cc16", label="Actual", linewidth=1.5)
axes[0].plot(forecast["ds"], forecast["yhat"],
             color="#f59e0b", label="Forecast", linewidth=1.5, linestyle="--")
axes[0].fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
                     alpha=0.2, color="#f59e0b", label="95% CI")
axes[0].axvline(prophet_df["ds"].max(), color="red", linestyle=":", alpha=0.5, label="Forecast start")
axes[0].set_title("Prophet Forecast — Avocado Prices", fontsize=13, fontweight="bold")
axes[0].legend()
axes[0].grid(alpha=0.2)

# Seasonality component
seasonal = model.predict(future)[["ds","yearly"]]
axes[1].plot(seasonal["ds"].dt.dayofyear, seasonal["yearly"],
             color="#38bdf8", linewidth=2)
axes[1].set_title("Yearly Seasonality Component", fontsize=12)
axes[1].set_xlabel("Day of Year")
axes[1].set_ylabel("Price Effect ($)")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()